# 🚀 Модуль 11. Урок 2. Векторные базы данных: индексация, метаданные и фильтрация в ChromaDB

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Enlyl/data-analyst-2027/blob/main/course/module_11_rag_vector_db/lesson_02_vector_databases_chromadb.ipynb)

## 📌 Бизнес-контекст и методология
Векторная база данных хранит эмбеддинги вместе с исходным текстом и метаданными, позволяя выполнять быстрый поиск ближайших соседей (HNSW / ANN).
Критически важный паттерн: **гибридная фильтрация** (поиск по вектору + жесткий фильтр по метаданным `category = 'finance'`).

In [1]:
# 🚀 Универсальная инициализация окружения и данных
import os
import urllib.request
import sqlite3
import pandas as pd
import numpy as np

def ensure_dataset(filename, subfolder=""):
    """Автоматически находит датасет локально или скачивает с GitHub репозитория курса."""
    if os.path.exists(filename):
        return filename
    candidates = [
        os.path.join("course", "datasets", subfolder, filename),
        os.path.join("course", "datasets", filename),
        os.path.join("..", "datasets", subfolder, filename),
        os.path.join("..", "datasets", filename),
        os.path.join("..", "..", "course", "datasets", subfolder, filename),
        os.path.join("datasets", subfolder, filename),
        os.path.join("datasets", filename),
    ]
    for c in candidates:
        if os.path.exists(c):
            return c
    raw_base = "https://raw.githubusercontent.com/Enlyl/data-analyst-2027/main/course/datasets"
    url = f"{raw_base}/{subfolder}/{filename}" if subfolder else f"{raw_base}/{filename}"
    print(f"📥 Загрузка {filename} из репозитория курса...")
    urllib.request.urlretrieve(url, filename)
    print(f"✅ Датасет {filename} успешно готов!")
    return filename

# Загрузка ключевых датасетов
customers_csv = ensure_dataset("customers.csv", subfolder="raw")
orders_csv = ensure_dataset("orders.csv", subfolder="raw")
saas_csv = ensure_dataset("saas_users.csv", subfolder="raw")
print("🟢 Окружение готово к анализу данных.")


🟢 Окружение готово к анализу данных.


## 🔍 Разбор механики и демонстрация на реальных данных

In [2]:
import numpy as np

def cosine_similarity(v1, v2):
    return float(np.dot(v1, v2) / (np.linalg.norm(v1) * np.linalg.norm(v2)))

# Симуляция векторного хранилища с метаданными
class SimpleVectorDB:
    def __init__(self):
        self.documents = []
        
    def add(self, doc_id, text, vector, metadata):
        self.documents.append({
            'id': doc_id,
            'text': text,
            'vector': np.array(vector),
            'metadata': metadata
        })
        
    def query(self, query_vec, filter_meta=None, top_k=2):
        results = []
        for doc in self.documents:
            if filter_meta:
                match = all(doc['metadata'].get(k) == v for k, v in filter_meta.items())
                if not match:
                    continue
            sim = cosine_similarity(query_vec, doc['vector'])
            results.append((sim, doc))
        results.sort(key=lambda x: x[0], reverse=True)
        return results[:top_k]

db = SimpleVectorDB()
db.add("1", "Как считать LTV для подписочных сервисов", [0.8, 0.9, 0.1], {"domain": "finance"})
db.add("2", "Инструкция по настройке SSO авторизации", [0.1, 0.2, 0.9], {"domain": "it_sec"})
db.add("3", "Когортный анализ удержания клиентов (Retention)", [0.75, 0.85, 0.15], {"domain": "finance"})

q_res = db.query(np.array([0.8, 0.88, 0.1]), filter_meta={"domain": "finance"}, top_k=1)
print("Найденный документ:")
print(q_res[0][1]['text'], f"(Score: {q_res[0][0]:.3f})")


Найденный документ:
Как считать LTV для подписочных сервисов (Score: 1.000)


### 🎯 Практическое задание 1: Проверка работы фильтра метаданных
Убедитесь, что найденный документ принадлежит домену `'finance'`.

In [3]:
### ВАШ КОД ЗДЕСЬ
top_doc = q_res[0][1]
assert top_doc['metadata']['domain'] == 'finance'
print("✅ Тест успешно пройден: гибридный поиск отработал безупречно!")


✅ Тест успешно пройден: гибридный поиск отработал безупречно!


### 💼 Паттерн для аналитика
При индексации документации по метрикам обязательно добавляйте в метаданные: `department`, `author`, `last_updated_year`.